# ResearchPaper1 — N=50 Final Comparative Search Replication
## 400 Trajectories = 50 Seeds × 2 Tasks × 4 Methods

**Core Research Question**: *Does the LLM reasoning advantage on Dyck-4 replicate at N=50 while disappearing on Hidden FSM under strictly matched candidate compute?*

### Experimental Matrix:
- **Seeds**: Exactly 50 deterministic seeds (10 established benchmark seeds + 40 frozen replication seeds)
- **Tasks**: Dyck-4 (Hierarchical Grammar) + Hidden FSM (Markov Regular)
- **Methods**: Random Search + TPE Bayesian + Evolutionary (GA) + Local LLM Reasoning Agent
- **Hardware**: Dual NVIDIA Tesla T4 GPUs (Kaggle Cloud Accelerator)
- **LLM Optimizer**: `Qwen/Qwen2.5-7B-Instruct` (Local in-memory execution, 4-bit NF4 quantization, 0 external API calls)
- **Horizon**: $K=10$ candidate iterations per arm | 50 gradient steps per evaluation
- **Ablations**: DISABLED (saves compute, isolates main comparative benchmark)
- **Provenance**: STRICT provenance auditing (0 simulation, 100% verified CUDA runs)

In [ ]:
# ==============================================================================
# 1. GPU VERIFICATION
# ==============================================================================
print("=" * 80)
print("STEP 1 — VERIFYING KAGGLE GPU")
print("=" * 80)
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    "FATAL: CUDA GPU not detected. Enable GPU accelerator in Kaggle Settings."
)
print(
    f"CUDA Available: {torch.cuda.is_available()} | "
    f"GPU Count: {torch.cuda.device_count()} | "
    f"GPU 0: {torch.cuda.get_device_name(0)}"
)
if torch.cuda.device_count() >= 2:
    print(f"GPU 1: {torch.cuda.get_device_name(1)}")
else:
    print("WARNING: Fewer than 2 GPUs detected.")

In [ ]:
# ==============================================================================
# 2. CLEAN REPOSITORY SYNC
# ==============================================================================
import os, shutil
print("=" * 80)
print("STEP 2 — SYNCING LATEST REPOSITORY")
print("=" * 80)
if os.path.exists("researchpaper1"):
    shutil.rmtree("researchpaper1")
!git clone https://github.com/Goldypahal/researchpaper1.git
%cd researchpaper1
!git checkout main
!git pull --ff-only origin main
print("\nCurrent commit:")
!git rev-parse HEAD

In [ ]:
# ==============================================================================
# 3. INSTALL DEPENDENCIES
# ==============================================================================
print("=" * 80)
print("STEP 3 — INSTALLING DEPENDENCIES")
print("=" * 80)
!pip install -q optuna scipy matplotlib transformers accelerate bitsandbytes

In [ ]:
# ==============================================================================
# 4. VERIFY EXACT 50 DETERMINISTIC SEEDS CONFIGURATION
# ==============================================================================
print("=" * 80)
print("STEP 4 — VERIFYING 50-SEED CONFIGURATION")
print("=" * 80)
runner = "run_on_kaggle.py"
with open(runner, "r", encoding="utf-8") as f:
    source = f.read()

# Verify native 50-seed deterministic expansion
assert "additional_seeds" in source, "FATAL: 50-seed expansion logic missing from runner."
assert "assert len(seeds) == num_seeds" in source, "FATAL: Seed count assertion missing."

standard_seeds = [42, 101, 202, 303, 404, 505, 606, 707, 808, 909]
additional_seeds = [1000 + 1009 * i for i in range(40)]
seeds = standard_seeds + additional_seeds
assert len(seeds) == 50 and len(set(seeds)) == 50
print("[PASS] Exactly 50 unique deterministic seeds verified:")
print(f"  First 10 (Benchmark): {seeds[:10]}")
print(f"  Additional 40 (Replication): {seeds[10:15]} ... {seeds[-5:]}")

In [ ]:
# ==============================================================================
# 5. REMOVE STALE EXPERIMENTAL TRACES
# ==============================================================================
print("=" * 80)
print("STEP 5 — REMOVING STALE TRACES FOR CLEAN UNCONTAMINATED SWEEP")
print("=" * 80)
trace_dirs = [
    "v0.3-comparative-search/traces",
    "v0.4-reasoning-ablation/traces",
]
for d in trace_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed: {d}")
os.makedirs("v0.3-comparative-search/traces", exist_ok=True)

for stale in ["kaggle_experiment_results.zip"]:
    if os.path.exists(stale):
        os.remove(stale)
        print(f"Removed: {stale}")

In [ ]:
# ==============================================================================
# 6. RUN N=50 MAIN COMPARATIVE EXPERIMENT (400 TRAJECTORIES)
# ==============================================================================
print("=" * 80)
print("STEP 6 — STARTING N=50 COMPARATIVE SEARCH REPLICATION")
print("=" * 80)
print("""
Experimental Design Checklist
-----------------------------
Seeds: 50 unique deterministic seeds
Tasks: Dyck-4 + Hidden FSM
Methods: Random + TPE + Evolutionary + LLM
Total Trajectories: 400
Horizon: K=10 candidate proposals per arm
Steps: 50 gradient steps per candidate
LLM Model: Qwen/Qwen2.5-7B-Instruct (4-bit NF4 quantized)
Provider: local (offline dual T4 execution)
Ablations: DISABLED (--no_ablations)
Simulation: DISABLED (STRICT provenance mode)
""")

!python run_on_kaggle.py \
    --seeds 50 \
    --iterations 10 \
    --steps 50 \
    --provider local \
    --model Qwen/Qwen2.5-7B-Instruct \
    --quantization 4bit \
    --no_ablations

In [ ]:
# ==============================================================================
# 7. AUDITING & SCIENTIFIC INTEGRITY VERIFICATION
# ==============================================================================
import glob, json
from collections import Counter

print("=" * 80)
print("STEP 7 — SCIENTIFIC INTEGRITY & COMPLETENESS AUDIT")
print("=" * 80)

trace_dir = "v0.3-comparative-search/traces"
trace_files = glob.glob(os.path.join(trace_dir, "trace_*.json"))
print(f"Total raw traces found: {len(trace_files)}")
assert len(trace_files) == 400, f"FATAL: Expected 400 traces, found {len(trace_files)}"
print("[PASS] Exactly 400 trajectories found.")

# Factorial Completeness Verification: 2 tasks x 4 methods x 50 seeds
tasks = ["dyck", "fsm"]
methods = ["random", "tpe", "evolutionary", "llm"]
standard_seeds = [42, 101, 202, 303, 404, 505, 606, 707, 808, 909]
seeds = standard_seeds + [1000 + 1009 * i for i in range(40)]
expected = {f"trace_{t}_{m}_seed_{s}.json" for t in tasks for m in methods for s in seeds}
actual = {os.path.basename(x) for x in trace_files}
missing = sorted(expected - actual)
assert not missing, f"FATAL: Missing traces: {missing[:10]}"
print("[PASS] Complete 2 x 4 x 50 factorial design verified.")

# Verify zero simulation markers in LLM traces
llm_files = [x for x in trace_files if "_llm_" in os.path.basename(x)]
assert len(llm_files) == 100, f"FATAL: Expected 100 LLM traces, found {len(llm_files)}"
simulation_hits = []
for fp in llm_files:
    with open(fp, "r", encoding="utf-8") as f:
        txt = f.read().lower()
        if "adaptive_simulation" in txt or "simulation mode" in txt or "scripted simulation" in txt:
            simulation_hits.append(fp)
assert not simulation_hits, f"FATAL: Simulation markers detected: {simulation_hits}"
print("[PASS] All 100 LLM trajectories are genuine local GPU executions.")

# Validate JSON format of every single trace
for fp in trace_files:
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)
        assert isinstance(data, dict)
print("[PASS] All 400 trace files parsed successfully as valid JSON.")

# Balanced Design Check
task_counts = Counter()
method_counts = Counter()
for fp in trace_files:
    name = os.path.basename(fp)
    for t in tasks:
        if f"trace_{t}_" in name:
            task_counts[t] += 1
    for m in methods:
        if f"_{m}_" in name:
            method_counts[m] += 1

assert task_counts["dyck"] == 200 and task_counts["fsm"] == 200
assert all(method_counts[m] == 100 for m in methods)
print("[PASS] Perfectly balanced: 200 Dyck + 200 FSM | 100 per search arm.")

In [ ]:
# ==============================================================================
# 8. DISPLAY REGENERATED PUBLICATION FIGURES
# ==============================================================================
from IPython.display import Image, display
print("=" * 80)
print("STEP 8 — RENDERING PUBLICATION FIGURES")
print("=" * 80)
fig_files = sorted(glob.glob("figures/fig*.png"))
if fig_files:
    for f in fig_files:
        print(f"\n--- {f} ---")
        display(Image(filename=f, width=850))
else:
    print("No figures found. Run 'python make_experiment_plots.py' manually if needed.")

In [ ]:
# ==============================================================================
# 9. EXPORT COMPLETE ZIP ARCHIVE
# ==============================================================================
print("=" * 80)
print("STEP 9 — EXPORTING COMPLETE ARTIFACT ARCHIVE")
print("=" * 80)
src_zip = "kaggle_experiment_results.zip"
dst_zip = "/kaggle/working/kaggle_experiment_results.zip"
if os.path.exists(src_zip):
    shutil.copy2(src_zip, dst_zip)
    size_mb = os.path.getsize(dst_zip) / (1024 ** 2)
    print(f"[SUCCESS] Archive ready: {dst_zip} ({size_mb:.2f} MB)")
    print("Download 'kaggle_experiment_results.zip' from the right-hand Kaggle Output sidebar!")
else:
    print("[WARNING] kaggle_experiment_results.zip was not found.")

print("\n" + "=" * 80)
print("N=50 EXPERIMENTAL RUN COMPLETE")
print("=" * 80)
print("""
Next Steps for Analysis:
1. Download 'kaggle_experiment_results.zip'.
2. Extract into local repository.
3. Run statistical aggregation across all 400 raw traces.
4. Compare Dyck-4 vs FSM separation at N=50.
""")